In [ ]:
import os
import sys
import pandas as pd
from prophet import Prophet
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.orm import Session
from sqlalchemy.dialects.postgresql import insert

#joins the project root to ensure models imports work 
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
from models.daily_air_forecast import DailyAirForecast

In [ ]:
load_dotenv()
SYNC_DB_URL = os.getenv('DB_URL').replace("+asyncpg", "")
engine = create_engine(SYNC_DB_URL)

In [ ]:
query = "SELECT * FROM historical_particles ORDER BY date;"
df = pd.read_sql(query, engine)
df['date'] = pd.to_datetime(df['date'])

In [ ]:
municipalities = df['municipality'].unique()
pollutants = ['no2', 'o3', 'co', 'so2']
units = {
    'no2': 'µg/m³',
    'o3': 'µg/m³', 
    'co': 'mg/m³', 
    'so2': 'µg/m³'
    }

In [ ]:
for muni in municipalities:
    muni_data = df[df['municipality'] == muni]
    
    for pol in pollutants:
        #Prepares data for Prophet
        model_data = muni_data[['date', pol]].copy()
        model_data = model_data.rename(columns={'date': 'ds', pol: 'y'}).dropna()
        
        #Skips if a region lacks historical data
        if len(model_data) < 30:
            continue
            
        #Initializes and trains Prophet
        m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
        m.fit(model_data)
        
        #Forecast for the next 7 days
        future = m.make_future_dataframe(periods=7)
        forecast = m.predict(future)
        
        last_7_actual = model_data.tail(7)
        next_7_pred = forecast.tail(7)
        
        #Calculates frontend UI metrics
        current_value = float(last_7_actual['y'].iloc[-1])
        predicted_peak = float(next_7_pred['yhat'].max())
        
        past_avg = last_7_actual['y'].mean()
        future_avg = next_7_pred['yhat'].mean()
        
        #Higher pollution is worse
        if future_avg > past_avg * 1.05:
            trend = "Worsening"
        elif future_avg < past_avg * 0.95:
            trend = "Improving"
        else:
            trend = "Stable"
            
        #Calculates confidence score based on Prophet's uncertainty intervals
        raw_margin = (next_7_pred['yhat_upper'] - next_7_pred['yhat_lower']) / next_7_pred['yhat']
        dampened_penalty = raw_margin.mean() / 3
        confidence = float(max(0.60, min(0.98, 1 - dampened_penalty)))
        
        #Formats arrays for Chart.js
        hist_dates = last_7_actual['ds'].dt.strftime('%Y-%m-%d').tolist()
        hist_data = [round(val, 2) for val in last_7_actual['y'].tolist()]
        fut_dates = next_7_pred['ds'].dt.strftime('%Y-%m-%d').tolist()
        fut_data = [max(0, round(val, 2)) for val in next_7_pred['yhat'].tolist()] 
        
        stmt = insert(DailyAirForecast).values(
            municipality=muni,
            pollutant=pol,
            current_value=round(current_value, 2),
            predicted_peak=round(predicted_peak, 2),
            trend=trend,
            confidence_score=round(confidence, 2),
            unit=units[pol],
            historical_dates=hist_dates,
            historical_data=hist_data,
            future_dates=fut_dates,
            future_data=fut_data
        )
        
        #Excludes the primary keys from being updated during a conflict
        update_dict = {c.name: c for c in stmt.excluded if c.name not in ['municipality', 'pollutant']}
        
        upsert_stmt = stmt.on_conflict_do_update(
            index_elements=['municipality', 'pollutant'],
            set_=update_dict
        )
        
        with Session(engine) as session:
            session.execute(upsert_stmt)
            session.commit()